## HW NN Model Calibration

In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import grad
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
from joblib import dump
import QuantLib as ql
from scipy.optimize import minimize
from py_vollib.black_scholes.implied_volatility import implied_volatility
import time

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load Data

In [ ]:
base_filename = 'HW_option_1_'

In [ ]:
hw_data_list = list()
for i in range(0,100):
    filename = base_filename + str(i) + '.csv'
    if os.path.exists(filename):
        new_hw_data = pd.read_csv(filename, index_col=[0])
        hw_data_list.append(new_hw_data) 
    
hw_data = pd.concat(hw_data_list)
hw_data.reset_index(inplace=True)

In [ ]:
hw_data

In [ ]:
y = hw_data[['a', 'sigma']].to_numpy()

In [ ]:
X = hw_data[['rate', '1, 5', '2, 4', '3, 3', '4, 2', '5, 1']]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=42)

In [ ]:
standard_scaler = StandardScaler()
sX_train = standard_scaler.fit_transform(X_train)
sX_test = standard_scaler.transform(X_test)

In [ ]:
output_scaler = MinMaxScaler()
sy_train = output_scaler.fit_transform(y_train)
sy_test = output_scaler.transform(y_test)

In [ ]:
sX_test

In [ ]:
sy_test

In [ ]:
dump(standard_scaler, 'hw_scaler.bin', compress=True)
dump(output_scaler, 'hw_scaler_out.bin', compress=True)

In [ ]:
batch_size = 1024

In [ ]:
train_x = torch.Tensor(sX_train).to(device)
train_y = torch.Tensor(sy_train).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(sX_test).to(device)
test_y = torch.Tensor(sy_test).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

## Train Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    grad_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        for batch_X, batch_y in train_loader:

            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)

        # Evaluation on the test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.6f}, \
                                   Test Loss: {test_loss:.6f}")
    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, width = 32, use_batch_norm=False):
        super(NeuralNetwork, self).__init__()
        self.use_batch_norm = use_batch_norm
        self.activation = Swish()
        self.fc1 = nn.Linear(6, width)
        self.bn1 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc2 = nn.Linear(width, width)
        self.bn2 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc3 = nn.Linear(width, width)
        self.bn3 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc4 = nn.Linear(width, width)
        self.bn4 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc5 = nn.Linear(width, 2)

    def forward(self, x):
        x = self.activation(self.bn1(self.fc1(x)))
        x = self.activation(self.bn2(self.fc2(x)))
        x = self.activation(self.bn3(self.fc3(x)))
        x = self.activation(self.bn4(self.fc4(x)))
        #x = torch.sigmoid(self.fc5(x))
        x = self.fc5(x)
        return x

In [ ]:
no_epochs = 100
loss_fn = nn.MSELoss()

In [ ]:
hwnn = NeuralNetwork(32, True).to(device)
optimizer = optim.Adam(hwnn.parameters(), lr=0.0001)
history = train_model(hwnn, train_loader, test_loader, 
                          loss_fn, optimizer, no_epochs)

In [ ]:
model_path = "hwnn.pth"

In [ ]:
torch.save(hwnn.state_dict(), model_path)

### Plot Losses

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
epochs

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

train_loss_values = history['train_loss']
test_loss_values = history['test_loss']
ax.plot(epochs, train_loss_values, color='black', linestyle='-',label = 'training loss', )
ax.plot(epochs, test_loss_values, color='black', linestyle='--', label = 'test loss')

ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()
fig.savefig('HWNNLoss.png', dpi=300, bbox_inches='tight')

In [ ]:
results = hwnn(test_x)

In [ ]:
results

In [ ]:
test_y

### Compare with direct calibration

In [ ]:
def calculate_implied_volatility(swaption_expiry, underlying_swap_tenor, flat_rate, strike, european_swaption_price):
    # Setup the calculation date and calendar
    calendar = ql.TARGET()
    settlement_date = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = settlement_date

    # Create the yield term structure
    rate_handle = ql.YieldTermStructureHandle(ql.FlatForward(settlement_date, ql.QuoteHandle(ql.SimpleQuote(flat_rate)), ql.Actual365Fixed()))

    # Define the dates for the swap
    start_date = calendar.advance(settlement_date, swaption_expiry, ql.Years)
    end_date = calendar.advance(start_date, underlying_swap_tenor, ql.Years)

    # Define the fixed leg of the swap
    fixed_leg_tenor = ql.Period(1, ql.Years)
    fixed_schedule = ql.Schedule(start_date, end_date, fixed_leg_tenor, calendar, ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

    # Define the floating leg of the swap
    float_leg_tenor = ql.Period(1, ql.Years)
    float_schedule = ql.Schedule(start_date, end_date, float_leg_tenor, calendar, ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

    # Define the underlying swap
    index = ql.Euribor1Y(rate_handle)
    swap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1.0, fixed_schedule, strike, ql.Actual360(), float_schedule, index, 0.0, index.dayCounter())

    # Define the swaption
    exercise = ql.EuropeanExercise(start_date)
    swaption = ql.Swaption(swap, exercise)

    # Setup the solver to find the implied volatility
    def error_function(vol):
        quote_handle = ql.QuoteHandle(ql.SimpleQuote(vol))
        engine = ql.BlackSwaptionEngine(rate_handle, quote_handle)
        swaption.setPricingEngine(engine)
        return swaption.NPV() - european_swaption_price

    # Solve for the implied volatility
    implied_vol = ql.Brent().solve(error_function, 1e-8, 0.2, 1e-8)
    return implied_vol

In [ ]:
calculate_implied_volatility(1, 5, 0.05, 0.05, 0.11)

In [ ]:
def create_swaptions(swaption_defs, vols, index, term_structure, engine):
    swaptions = []
    fixed_leg_tenor = ql.Period(1, ql.Years)
    fixed_leg_daycounter = ql.Actual360()
    floating_leg_daycounter = ql.Actual360()
    for i, d in enumerate(swaption_defs):
        vol_handle = ql.QuoteHandle(ql.SimpleQuote(vols[i]))
        helper = ql.SwaptionHelper(ql.Period(d[0], ql.Years),
                                   ql.Period(d[1], ql.Years),
                                   vol_handle,
                                   index,
                                   fixed_leg_tenor,
                                   fixed_leg_daycounter,
                                   floating_leg_daycounter,
                                   term_structure
                                   )
        helper.setPricingEngine(engine)
        swaptions.append(helper)
    return swaptions   

In [ ]:
ex_idx = 3

In [ ]:
swaptions_defs = [[1, 5], [2, 4], [3, 3], [4, 2], [5, 1]]
swaptions_strs = ['1, 5', '2, 4', '3, 3', '4, 2', '5, 1']

In [ ]:
flat_rate = X_test['rate'].iloc[ex_idx]
flat_rate

In [ ]:
sw_prices = X_test[swaptions_strs].iloc[ex_idx]
sw_prices

In [ ]:
sw_imp_vols = []
for i, idef in enumerate(swaptions_defs):
    price = sw_prices.iloc[i]
    imp_vol = calculate_implied_volatility(idef[0], idef[1], flat_rate, flat_rate, price)
    print(imp_vol)
    sw_imp_vols.append(imp_vol)

In [ ]:
today = ql.Date.todaysDate()
ql.Settings.instance().evaluationDate = today
rate_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, 
                                                         flat_rate, 
                                                         ql.Actual365Fixed()))
index = ql.Euribor1Y(rate_handle)

In [ ]:
model = ql.HullWhite(rate_handle);
engine = ql.JamshidianSwaptionEngine(model)
swaptions = create_swaptions(swaptions_defs, sw_imp_vols, index, rate_handle, engine)

optimization_method = ql.LevenbergMarquardt(1.0e-8,1.0e-8,1.0e-8)
end_criteria = ql.EndCriteria(1000, 100, 1e-6, 1e-8, 1e-8)
model.calibrate(swaptions, optimization_method, end_criteria)

a, sigma = model.params()

In [ ]:
print(model.params())

In [ ]:
output_scaler.inverse_transform(test_y.cpu().numpy())[ex_idx]

In [ ]:
output_scaler.inverse_transform(results.detach().cpu().numpy())[ex_idx]

In [ ]:
test_param = output_scaler.inverse_transform(test_y.cpu().numpy())

In [ ]:
dnn_param = output_scaler.inverse_transform(results.detach().cpu().numpy())

In [ ]:
residuals = test_param - dnn_param

In [ ]:
rel_residuals = residuals / test_param

In [ ]:
rel_residuals

### Quiver Plot

In [ ]:
# Extract X and Y coordinates for ground truth vectors
X = test_param[:, 0]
Y = test_param[:, 1]

# Extract X and Y components of error vectors
U = residuals[:, 0]
V = residuals[:, 1]

# Create quiver plot
plt.figure()
plt.quiver(X, Y, U, V, angles='xy', scale_units='xy', scale=1, color='r')
plt.xlim(0, np.max(X+U) + 1)
plt.ylim(0, np.max(Y+V) + 1)
plt.xlabel('a')
plt.ylabel('sigma')
plt.grid()
plt.savefig('HWNNQiver.png', dpi=300, bbox_inches='tight')

### Bland-Altman Plot

In [ ]:
# Calculate magnitudes of vectors
ground_truth_magnitudes = np.linalg.norm(test_param, axis=1)
test_magnitudes = np.linalg.norm(dnn_param, axis=1)

means = (ground_truth_magnitudes + test_magnitudes) / 2
differences = test_magnitudes - ground_truth_magnitudes

# Calculate the average difference (bias) and the limits of agreement
mean_difference = np.mean(differences)
std_difference = np.std(differences)
upper_limit = mean_difference + 1.96 * std_difference
lower_limit = mean_difference - 1.96 * std_difference

# Create the Bland-Altman plot
plt.figure()
plt.scatter(means, differences, color='blue')
plt.axhline(mean_difference, color='red', linestyle='--')
plt.axhline(upper_limit, color='green', linestyle='--')
plt.axhline(lower_limit, color='green', linestyle='--')
plt.text(np.max(means), mean_difference, 'Mean', va='bottom', ha='right')
plt.text(np.max(means), upper_limit, 'Upper Limit of Agreement', va='bottom', ha='right')
plt.text(np.max(means), lower_limit, 'Lower Limit of Agreement', va='bottom', ha='right')
plt.xlabel('Mean Magnitude')
plt.ylabel('Difference in Magnitude')
plt.grid(True)
plt.savefig('HWNNBlandAltamn.png', dpi=300, bbox_inches='tight')

### Heatmap

In [ ]:
# Calculate error vectors and their magnitudes
error_vectors = (test_param - dnn_param)
error_magnitudes = np.linalg.norm(error_vectors, axis=1)

# Define the bin edges for the 2D space
x_edges = np.linspace(0, 0.5, 51)  # Adjust bin sizes and counts as needed
y_edges = np.linspace(0, 0.05, 51)

# Create an empty grid to store average error magnitudes
error_grid = np.zeros((len(x_edges) - 1, len(y_edges) - 1))

# Bin the data and calculate the average error in each bin
for i in range(len(x_edges) - 1):
    for j in range(len(y_edges) - 1):
        # Find points within the current bin
        indices = np.where(
            (test_param[:, 0] >= x_edges[i]) & 
            (test_param[:, 0] < x_edges[i + 1]) & 
            (test_param[:, 1] >= y_edges[j]) & 
            (test_param[:, 1] < y_edges[j + 1])
        )[0]
        
        # Calculate the average error for points in the bin
        if len(indices) > 0:
            average_error = np.mean(error_magnitudes[indices])
        else:
            average_error = np.nan  # No data in this bin
        
        error_grid[i, j] = average_error

# Plotting the heatmap
plt.figure(figsize=(8, 6))
n = 10
xticklab = [f"{x:.1f}" if i % n == 0 else '' for i, x in enumerate(x_edges)]
yticklab = [f"{y:.2f}" if i % n == 0 else '' for i, y in enumerate(y_edges)]
sns.heatmap(error_grid, xticklabels=xticklab, yticklabels=yticklab, cmap='Reds', cbar_kws={'label': 'Average Error Magnitude'})
plt.xlabel('a')
plt.ylabel('sigma')
plt.gca().invert_yaxis()
current_axis = plt.gca()
current_axis.add_patch(Rectangle((0, 0), len(x_edges) - 1, len(y_edges) - 1, fill=None, edgecolor='black', linewidth=2))

plt.savefig('HWNNHeatMap.png', dpi=300, bbox_inches='tight')

In [ ]:
# Plotting the heatmap
plt.figure(figsize=(8, 6))
n = 10
xticklab = [f"{x:.1f}" if i % n == 0 else '' for i, x in enumerate(x_edges)]
yticklab = [f"{y:.2f}" if i % n == 0 else '' for i, y in enumerate(y_edges)]
sns.heatmap(error_grid, xticklabels=xticklab, yticklabels=yticklab, cmap='Greys', cbar_kws={'label': 'Average Error Magnitude'})

plt.xlabel('a')
plt.ylabel('sigma')
plt.gca().invert_yaxis()
current_axis = plt.gca()
current_axis.add_patch(Rectangle((0, 0), len(x_edges) - 1, len(y_edges) - 1, fill=None, edgecolor='black', linewidth=2))
plt.savefig('HWNNHeatMapBW.png', dpi=300, bbox_inches='tight')